# 🚀 Hybrid RAG System with Cross-Encoder Reranking & Evaluation

An interview-grade, production-style Retrieval-Augmented Generation (RAG) system designed to eliminate retrieval failure modes on enterprise documents.

---

## 🏗️ Architecture Overview

```text
                               PDF Documents
                                     │
                                     ▼
                            [ 1. PDF Loading ]
                        (Extract text + page metadata)
                                     │
                                     ▼
                           [ 2. Text Chunking ]
                   (Split into small, coherent passages)
                                     │
                    ┌────────────────┴────────────────┐
                    ▼                                 ▼
         [ 3A. Dense Retrieval ]             [ 3B. BM25 Retrieval ]
       • Sentence/Nomic Embeddings         • Term frequency / IDF
       • Stored in Pinecone Vector DB      • Exact keywords, codes, names
       • Finds: 'hardware vs services'     • Finds: 'Q1 2024', '$119.58B'
                    │                                 │
                    └────────────────┬────────────────┘
                                     ▼
                      [ 4. Reciprocal Rank Fusion (RRF) ]
                       Merge & deduplicate candidate pools
                                     │
                                     ▼
                     [ 5. Cross-Encoder Reranking ]
                      (FlashRank / Cross-Encoder)
                  Jointly scores (Query, Chunk) pairs
                                     │
                                     ▼
                         [ Top-K Selected Chunks ]
                                     │
                                     ▼
                          [ 6. Prompt + Context ]
                                     │
                                     ▼
                          [ 7. Local Ollama LLM ]
                               (llama3.2:3b)
                                     │
                                     ▼
                      [ 8. Grounded Answer + Sources ]
                                     │
                    ┌────────────────┴────────────────┐
                    ▼                                 ▼
         [ 9A. Retrieval Precision@K ]      [ 9B. Faithfulness Eval ]
          Did we retrieve the right pages?   Is the answer backed by text?
```

---

## Section 1: Why Hybrid RAG? (The Problem We Are Solving)

Most beginner RAG tutorials only implement **Dense Vector Search** (`vectorstore.similarity_search()`). While embeddings are great at finding conceptual similarity, they fail in two common real-world scenarios:

1. **Exact Numbers & Financial Codes:** Dense embeddings compress an entire 500-token paragraph into 768 floating-point numbers. In this compression, specific numbers (e.g., `$119.58 billion` vs `$117.15 billion`) or fiscal quarters (`Q1 2024` vs `Q1 2023`) get blurred into general "financial revenue" vectors.
2. **Specific Term Matching:** Alphanumeric codes, rare product SKUs, or section titles often have poor representation in generic embedding spaces.

### The Solution:
* **BM25 (Sparse Lexical Search):** Directly scores exact word matches using Term Frequency (TF) and Inverse Document Frequency (IDF). It never misses an exact keyword.
* **Dense Search:** Understands synonyms and high-level concepts.
* **Reciprocal Rank Fusion (RRF):** Combines the ranks from both methods safely.
* **Cross-Encoder Reranking:** Re-scores the top candidates with full attention before passing them to the LLM.

## Section 2: Environment & Dependency Verification

In this section, we load our environment variables, verify our local Ollama instance (`llama3.2:3b`), and test our connection to the Pinecone cloud vector database.

In [22]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "apple-hybrid-rag")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")

print("✅ Environment configuration loaded:")
print(f"  • Pinecone Index Name: {PINECONE_INDEX_NAME}")
print(f"  • Pinecone API Key set: {'Yes (length ' + str(len(PINECONE_API_KEY)) + ')' if PINECONE_API_KEY else '❌ NOT SET'}")
print(f"  • Ollama Base URL: {OLLAMA_BASE_URL}")
print(f"  • Ollama Model: {OLLAMA_MODEL}")

✅ Environment configuration loaded:
  • Pinecone Index Name: apple-hybrid-rag
  • Pinecone API Key set: Yes (length 75)
  • Ollama Base URL: http://localhost:11434
  • Ollama Model: llama3.2:3b


In [23]:
import sys
print(sys.executable)          # what interpreter is REALLY running?
import pinecone; print("pinecone", pinecone.__version__)   # can it see our package?


c:\Users\ASUS\Desktop\RAG-Project2\temp\Scripts\python.exe
pinecone 10.0.0


In [24]:
# Verify Ollama Local LLM Connection
from langchain_ollama import ChatOllama

try:
    llm = ChatOllama(
        model=OLLAMA_MODEL,
        temperature=0.2,
        base_url=OLLAMA_BASE_URL
    )
    test_response = llm.invoke("Respond with 'Ollama is ready!' in 3 words.")
    print("✅ Ollama Connection Successful!")
    print("  Model Response:", test_response.content.strip())
except Exception as e:
    print("❌ Ollama connection failed. Make sure Ollama is running (`ollama serve`):")
    print("  Error:", e)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


✅ Ollama Connection Successful!
  Model Response: Ollama is ready!


In [25]:
# Verify Pinecone Connection
from pinecone import Pinecone

try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
    indexes = pc.list_indexes().names()
    print("✅ Pinecone Connection Successful!")
    print(f"  Existing indexes in your account: {indexes}")
except Exception as e:
    print("❌ Pinecone connection failed. Check your PINECONE_API_KEY in .env:")
    print("  Error:", e)

INFO:pinecone.client.indexes:Listing indexes
INFO:httpx:HTTP Request: GET https://api.pinecone.io/indexes "HTTP/1.1 200 OK"


✅ Pinecone Connection Successful!
  Existing indexes in your account: ['apple-hybrid-rag', 'hr-policy-index']


## Section 3: Load Documents & Inspect Metadata

A PDF is a binary container — we must extract its text before the pipeline can use it.
`PyPDFLoader` reads each PDF and returns **one `Document` per page**.

A `Document` has two fields:
- `page_content` → the extracted text
- `metadata` → `{"source": "file.pdf", "page": N}` — the provenance that will power citations later

```text
PDF Files → [PDF Loading] → Chunking → Dense + BM25 → RRF → Reranker → LLM
                             ▲ you are here



In [26]:
from langchain_community.document_loaders import PyPDFLoader

PDF_FILES = [
    "data/apple_fy24_q1.pdf",
    "data/apple_fy23_q4.pdf",
    "data/apple_fy23_q3.pdf",
]

documents = []
for path in PDF_FILES:
    loader = PyPDFLoader(path)
    for doc in loader.load():
        doc.page_content = doc.page_content.replace("/dollarsign", "$")
        documents.append(doc)

print(f"Loaded {len(documents)} page-documents from {len(PDF_FILES)} PDFs")


Loaded 9 page-documents from 3 PDFs


Section 4 — Text Chunking & Inspection



In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
chunks = splitter.split_documents(documents)

first = {}                                        # propagate page headers into every chunk
for cid, c in enumerate(chunks):
    key = (c.metadata["source"], c.metadata["page"])
    if key in first and cid != first[key]:
        c.page_content = chunks[first[key]].page_content + "\n" + c.page_content
    else:
        first[key] = cid

print(f"Split {len(documents)} pages into {len(chunks)} chunks (headers propagated)")


Split 9 pages into 61 chunks (headers propagated)


Step -5.1 Embegging of chunks 


In [28]:
from langchain_ollama import OllamaEmbeddings

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text:latest")

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

sample = embeddings.embed_query("What were Apple's net sales in Q1 2024?")
print("Embedding dimension:", len(sample))
print("Sample values:", [round(x, 4) for x in sample[:5]])


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Embedding dimension: 768
Sample values: [-0.0022, 0.0614, -0.2077, 0.0033, 0.0117]


Step 5.2- Index,embed , upsert(cell A
)

In [29]:
import time
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(name=PINECONE_INDEX_NAME, dimension=768, metric="cosine")
    while not pc.describe_index(PINECONE_INDEX_NAME).status.ready:
        time.sleep(1)
    print("Index created:", PINECONE_INDEX_NAME)
else:
    print("Index already exists:", PINECONE_INDEX_NAME)

index = pc.Index(PINECONE_INDEX_NAME)

vectors = []
for i, chunk in enumerate(chunks):
    vec = embeddings.embed_documents([chunk.page_content])[0]
    vectors.append((f"chunk-{i}", vec, chunk.metadata))

index.upsert(vectors=vectors)
print(f"Upserted {len(vectors)} vectors into {PINECONE_INDEX_NAME}")


INFO:pinecone.client.indexes:Listing indexes
INFO:httpx:HTTP Request: GET https://api.pinecone.io/indexes "HTTP/1.1 200 OK"
INFO:pinecone.client.indexes:Describing index 'apple-hybrid-rag'


Index already exists: apple-hybrid-rag


INFO:httpx:HTTP Request: GET https://api.pinecone.io/indexes/apple-hybrid-rag "HTTP/1.1 200 OK"
INFO:pinecone.index:Index client created for host https://apple-hybrid-rag-ztwp3zo.svc.aped-4627-b74a.pinecone.io
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embe

Upserted 61 vectors into apple-hybrid-rag


Step 5.3 — First semantic query (Cell B)


In [30]:
query_vec = embeddings.embed_query("What were Apple's net sales in Q1 2024?")
results = index.query(vector=query_vec, top_k=3, include_metadata=True)

for match in results.matches:
    i = int(match.id.split("-")[1])
    print(f"{match.score:.4f} | {match.metadata['source']} page {match.metadata['page']} | {chunks[i].page_content[:80]!r}")


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3


0.8072 | data/apple_fy24_q1.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) \n(In mil'
0.8048 | data/apple_fy24_q1.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) \n(In mil'
0.8037 | data/apple_fy23_q4.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) \n(In mil'


## Section 6: BM25 — Lexical Retrieval (The Sparse Arm)

Dense search understands *meaning*; BM25 understands *exact words*.

BM25 scores each chunk from two counts:

- **Term Frequency (TF):** how often a query word appears *in the chunk*
- **Inverse Document Frequency (IDF):** how *rare* that word is across all chunks

Rare words ("iphone") get big weight; stopwords ("the", "of") get ~zero.
A chunk missing a query word scores 0. No blur, no synonyms — just counts.

```text
chunks → lowercased tokens → BM25 term-frequency index


In [31]:

from rank_bm25 import BM25Okapi

tokenized_corpus = [chunk.page_content.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=3):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(i, scores[i]) for i in ranked]


In [32]:
#demo
for query in ["net sales", "iphone"]:
    print("Query:", repr(query))
    for i, score in bm25_search(query, top_k=2):
        print(f"  {score:.2f} | {chunks[i].metadata['source']} page {chunks[i].metadata['page']} | {chunks[i].page_content[:60]!r}")


Query: 'net sales'
  3.54 | data/apple_fy24_q1.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS '
  3.23 | data/apple_fy24_q1.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS '
Query: 'iphone'
  2.39 | data/apple_fy23_q4.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS '
  2.37 | data/apple_fy23_q4.pdf page 0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS '


Dense VS BM25 Head-to-Head

In [33]:
def dense_search(query, top_k=3):
    qv = embeddings.embed_query(query)
    matches = index.query(vector=qv, top_k=top_k, include_metadata=True).matches
    return [(int(m.id.split("-")[1]), m.score) for m in matches]

def show(query, top_k=2):
    print("=" * 60)
    print("Query:", repr(query))
    print("-- DENSE --")
    for i, s in dense_search(query, top_k):
        print(f"  {s:.4f} | {chunks[i].metadata['source']} p{chunks[i].metadata['page']} | {chunks[i].page_content[:45]!r}")
    print("-- BM25 --")
    for i, s in bm25_search(query, top_k):
        print(f"  {s:.2f} | {chunks[i].metadata['source']} p{chunks[i].metadata['page']} | {chunks[i].page_content[:45]!r}")

for question in ["iphone", "net sales", "how did Apple's services business perform?"]:
    show(question)


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=2


Query: 'iphone'
-- DENSE --


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=2


  0.6069 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.5909 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
-- BM25 --
  2.39 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  2.37 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
Query: 'net sales'
-- DENSE --


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=2


  0.6380 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.6167 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
-- BM25 --
  3.54 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  3.23 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
Query: "how did Apple's services business perform?"
-- DENSE --
  0.7101 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.7101 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
-- BM25 --
  1.65 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  1.63 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'


## Section 7: Reciprocal Rank Fusion (RRF)

    RRF(chunk) = Σ over retrievers of  1 / (k + rank_of_chunk)

Two contributions of the formula:

1. `k` smooths the positional gap (explained in code)
2. A chunk ranked by BOTH retrievers collects TWO terms → it rises.
   A chunk only ONE retriever loved (like a title page) → diluted.

Hand-rolled here (no library) so the math stays visible.


In [34]:
def rrf(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)


In [35]:
#demo cell 
def merge(query, top_k=5):
    dense_ranks = [i for i, _ in dense_search(query, top_k)]
    bm25_ranks  = [i for i, _ in bm25_search(query, top_k)]
    return rrf([dense_ranks, bm25_ranks])

for question in ["iphone", "net sales", "how did Apple's services business perform?"]:
    print("=" * 60)
    print("Query:", repr(question))
    for i, score in merge(question)[:3]:
        print(f"  {score:.4f} | {chunks[i].metadata['source']} p{chunks[i].metadata['page']} | {chunks[i].page_content[:45]!r}")


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


Query: 'iphone'


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


  0.0325 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.0323 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.0318 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
Query: 'net sales'


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


  0.0320 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.0320 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.0320 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
Query: "how did Apple's services business perform?"
  0.0164 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.0164 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'
  0.0161 | data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STATEMENTS'


## Section 8: Cross-Encoder Reranking

Dense + BM25 = retrieval (fast, broad, rough)
RRF → candidate pool
Cross-Encoder = reranking (slow, precise, on the pool only)

A cross-encoder reads (query + passage) TOGETHER → joint relevance score.
Each pool chunk gets rescored; top-N carry the final ranking.


In [36]:
from flashrank import Ranker, RerankRequest

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")
print("Ranker ready")


Ranker ready


In [37]:
def rerank(query, candidates, top_n=3):
    passages = [{"text": chunks[i].page_content, "meta": i} for i in candidates]
    results = ranker.rerank(RerankRequest(query=query, passages=passages))
    return [(int(r["meta"]), float(r["score"]), idx + 1)
            for idx, r in enumerate(results[:top_n])]


for question in ["iphone", "net sales", "how did Apple's services business perform?"]:
    print("=" * 60)
    print("Query:", repr(question))
    candidates = [i for i, _ in merge(question)]                     # RRF pool (Section 7)
    print("-- RRF pool --")
    for i in candidates:
        print(f"   {chunks[i].metadata['source']} p{chunks[i].metadata['page']} | {chunks[i].page_content[:38]!r}")
    print("-- RERANKED top-3 --")
    for i, score, rank in rerank(question, candidates[:5]):
        print(f"   {score:.4f} | {chunks[i].metadata['source']} p{chunks[i].metadata['page']} | {chunks[i].page_content[:38]!r}")


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


Query: 'iphone'
-- RRF pool --
   data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy23_q3.pdf p1 | 'Apple Inc. \nCONDENSED CONSOLIDATED BAL'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
-- RERANKED top-3 --
   0.9871 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   0.9662 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   0.9595 | data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
Query: 'net sales'


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


-- RRF pool --
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy23_q4.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
-- RERANKED top-3 --
   0.9894 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   0.9869 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   0.9859 | data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
Query: "how did Apple's services business perform?"
-- RRF pool --
   data/apple_fy23_q3.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED STA'
   data/apple_fy24_q1.pdf p0 | 'Apple Inc. \nCONDENSED CONSOLIDATED

## Section 9: Grounded RAG Generation

The full chain: question → Dense+BM25 → RRF pool → rerank → top-3
→ PROMPT (= question + retrieved context) → Ollama (llama3.2:3b) → answer + sources

Two guardrails in the prompt:
1. "Answer ONLY from the context" — no external knowledge
2. "If not found, say you could not find sufficient information" — no guessing


In [45]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.2, base_url=OLLAMA_BASE_URL)

def answer(question, top_n=3):
    candidates = [i for i, _ in merge(question)]
    ranked = rerank(question, candidates[:5])[:top_n]

    blocks, sources = [], []
    for i, _, _ in ranked:
        blocks.append(f"[{len(sources) + 1}] {chunks[i].page_content}")
        sources.append(f"{chunks[i].metadata['source']} — page {chunks[i].metadata['page']}")

    prompt = f"""You are a financial analyst assistant. Answer ONLY from the context below.

CONTEXT:
{chr(10).join(blocks)}

QUESTION: {question}

Rules:
- If the context does not contain the answer, say "I could not find sufficient information".
- Cite each fact as [N] matching its context block.

ANSWER:"""

    response = llm.invoke(prompt)
    return {"question": question, "answer": response.content,
            "context": chr(10).join(blocks), "sources": sources}


## Section 10: Retrieval Evaluation — Precision@K

A small HAND-LABELED test set: (question → relevant chunk ids).
Retrieval precision @K = |retrieved∩relevant| / K.
We compare Dense vs BM25 vs RRF vs RRF+Reranker on the SAME questions.
No fabricated numbers — every result comes from running this notebook.


In [39]:
## Code cell 1 — the test set (ground truth from the corpus we just browsed)

TEST_SET = [
    ("What were Apple's total net sales for the three months ended December 30, 2023?", {1, 4}),
    ("What were Apple's iPhone net sales for the three months ended December 30, 2023?", {4}),
    ("Which region accounted for the highest net sales in the three months ended December 30, 2023?", {4}),
    ("What was Apple's net income for the three months ended December 30, 2023?", {3}),
    ("How did Apple's services revenue compare year over year in the three months ended December 30, 2023?", {1}),
    ("What were iPhone net sales in the quarter ended September 30, 2023?", {25}),
]


In [40]:
## Code cell 2 — metric + head-to-head runner
def precision_at_k(retrieved_ids, relevant_ids, k=3):
    return len(set(retrieved_ids[:k]) & relevant_ids) / k

def score_one(fn, question, relevant, k=3):
    ids = [i for i, *_ in fn(question)]          # ids only; scores ignored
    return precision_at_k(ids, relevant, k), ids

retrievers = {
    "Dense":      lambda q: dense_search(q, 3),
    "BM25":       lambda q: bm25_search(q, 3),
    "RRF":        lambda q: merge(q, 3),
    "RRF+Rerank": lambda q: rerank(q, [i for i, _ in merge(q, 5)], 3),
}

for name, fn in retrievers.items():
    totals = []
    print("=" * 60)
    for question, relevant in TEST_SET:
        p, ids = score_one(fn, question, relevant)
        totals.append(p)
        print(f"  {p:.2f}  {ids}  | {question[:48]}")
    print(f"  >>> {name} AVG P@3: {sum(totals)/len(totals):.2f}")


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3


  0.33  [0, 4, 18]  | What were Apple's total net sales for the three 


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3


  0.33  [4, 24, 0]  | What were Apple's iPhone net sales for the three


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3


...

  0.33  [4, 3, 0]  | Which region accounted for the highest net sales


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


  0.00  [11, 17, 14]  | What was Apple's net income for the three months


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5


  0.33  [1, 0, 3]  | How did Apple's services revenue compare year ov
  0.33  [24, 25, 18]  | What were iPhone net sales in the quarter ended 
  >>> RRF+Rerank AVG P@3: 0.28


In [41]:
# ===== EXPERIMENT: header propagation, A/B in namespace "v2" =====
from copy import deepcopy

chunks_h = deepcopy(chunks)                          # augmented copies — originals intact
first = {}
for cid, c in enumerate(chunks_h):
    key = (c.metadata["source"], c.metadata["page"])
    if key in first and cid != first[key]:
        c.page_content = chunks_h[first[key]].page_content + "\n" + c.page_content
    else:
        first[key] = cid

bm_h = BM25Okapi([c.page_content.lower().split() for c in chunks_h])   # instant re-index

vecs = []
for cid, c in enumerate(chunks_h):
    vecs.append((f"chunk-{cid}", embeddings.embed_documents([c.page_content])[0], c.metadata))
index.upsert(vectors=vecs, namespace="v2")            # baseline namespace "" stays as-is
print("Upserted", len(vecs), "augmented vectors to namespace 'v2'")

def eval_p3(ns, corpus, bm_model):
    def d(q, k=3):
        ms = index.query(vector=embeddings.embed_query(q), top_k=k, namespace=ns, include_metadata=True).matches
        return [int(m.id.split("-")[1]) for m in ms]
    def b(q, k=3):
        s = bm_model.get_scores(q.lower().split())
        return sorted(range(len(s)), key=lambda i: s[i], reverse=True)[:k]
    def fus(q, k=3):
        return [i for i, _ in rrf([d(q, k), b(q, k)])][:k]
    def rr(q, k=3):
        pool = fus(q, 5)
        ranked = ranker.rerank(RerankRequest(query=q, passages=[{"text": corpus[i].page_content, "meta": i} for i in pool]))
        return [int(x["meta"]) for x in ranked[:k]]
    out = {}
    for name, fn in [("Dense", d), ("BM25", b), ("RRF", fus), ("RRF+Rerank", rr)]:
        hits = [len(set(fn(q)) & rel) / 3 for q, rel in TEST_SET]
        out[name] = round(sum(hits) / len(hits), 2)
    return out

print("BASELINE P@3    :", eval_p3("", chunks, bm25))
print("HEADERS P@3 v2  :", eval_p3("v2", chunks_h, bm_h))


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POS

Upserted 61 augmented vectors to namespace 'v2'


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.in

BASELINE P@3    : {'Dense': 0.28, 'BM25': 0.22, 'RRF': 0.28, 'RRF+Rerank': 0.28}


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=3
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.in

HEADERS P@3 v2  : {'Dense': 0.22, 'BM25': 0.28, 'RRF': 0.28, 'RRF+Rerank': 0.28}


Section 11 — Faithfulness


In [48]:
import re

def figures(t):
    return {n.rstrip(",").rstrip(".") for n in re.findall(r"\d[\d,]*", t)}

def faithfulness(result):
    unsupported = figures(result["answer"]) - figures(result["context"])

    cited = set(re.findall(r"\[\d+\]", result["answer"]))
    allowed = {f"[{i}]" for i in range(1, len(result["sources"]) + 1)}
    bad_cites = cited - allowed

    ok = not unsupported and not bad_cites
    print("ANSWER:", result["answer"])
    print("Numbers unsupported by context:", sorted(unsupported) or "none ✔")
    print("Invalid citations:", sorted(bad_cites) or "none ✔")
    print("SOURCES:", result["sources"])
    return ok

for question in [
    "What were Apple's total net sales for the three months ended December 30, 2023?",
    "What were Apple's iPhone net sales for the three months ended December 30, 2023?",
    "How did Apple's services business perform in the three months ended December 30, 2023?",
]:
    print("=" * 60)
    faithful = faithfulness(answer(question))
    print("FAITHFUL ✔" if faithful else "UNFAITHFUL ✘")
    print()


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


ANSWER: According to [1], Apple's total net sales for the three months ended December 30, 2023, were $119,575 million.
Numbers unsupported by context: none ✔
Invalid citations: none ✔
SOURCES: ['data/apple_fy24_q1.pdf — page 0', 'data/apple_fy24_q1.pdf — page 0', 'data/apple_fy24_q1.pdf — page 0']
FAITHFUL ✔



INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


ANSWER: According to the context, Apple's iPhone net sales for the three months ended December 30, 2023, were $69,702 million. 

[N] Three Months Ended December 30, 2023 
Net sales:    
   Products $ 96,458   $ 96,388  
   Services  23,117    20,766  
Total net sales (1)  119,575    117,154  
[1] Net sales by category:    
iPhone $ 69,702   $ 65,775  
...
Numbers unsupported by context: none ✔
Invalid citations: none ✔
SOURCES: ['data/apple_fy24_q1.pdf — page 0', 'data/apple_fy24_q1.pdf — page 0', 'data/apple_fy24_q1.pdf — page 0']
FAITHFUL ✔



INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:pinecone.index:Querying index with top_k=5
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


ANSWER: According to the condensed consolidated statements of operations, Apple's services business generated $23,117 million in revenue for the three months ended December 30, 2023.

[N] Apple Inc. 
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) 
(In millions, except number of shares, which are reflected in thousands, and per-share amounts) 
 Three Months Ended 
 December 30, 2023  December 31, 2022 
Net sales:    
   Products $ 96,458   $ 96,388  
   Services  23,117    20,766  
Total net sales (1)  119,575    117,154  
[N] Apple Inc. 
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) 
(In millions, except number of shares, which are reflected in thousands, and per-share amounts) 
 Three Months Ended 
 December 30, 2023  December 31, 2022 
Net sales:    
   Products $ 96,458   $ 96,388  
   Services  23,117    20,766  
Total net sales (1)  119,575    117,154  
[N] Apple Inc. 
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) 
(In millions, except numb